# Parcel-Level IS-RSA — Left-Wing Subjects

**Hypothesis 3**: Individual affective polarization modulates pairwise neural similarity.
Subjects who are more similar in `affpol_thermo` should show more similar spatial
activation patterns in a given parcel when viewing political content.

**Behavioral score**: `affpol_thermo` — in-party thermometer − out-party thermometer (~[−100, 100]).

**Method (IS-RSA)** — for each parcel and condition:
1. Build N×N **neural similarity matrix** from pairwise Pearson r across voxels within the parcel.
2. Build N×N **behavioral similarity matrix** from pairwise `affpol_thermo` scores.
3. Correlate upper triangles → IS-RSA r per parcel.
4. Permutation test: shuffle subject labels on behavioral matrix (1,000 iterations).
5. FDR correction (Benjamini–Hochberg, q=0.05).

**Two approaches** (matching `parcel_ispc_leftwing.ipynb`):

| | Approach A | Approach B |
|---|---|---|
| Neural similarity entry (i,j) | Pearson r between subjects' **mean** patterns (averaged over posts) across voxels | **Per-post** Pearson r across voxels, averaged over posts |
| Asks | Stable condition-level spatial similarity? | Post-by-post spatial similarity? |
| SNR | Higher | Lower (per-post noise retained) |

**Three analysis levels**:
- **Level A** — All 4 conditions: AntiLeft · AntiRight · ProLeft · ProRight
- **Level B** — Agreed={AntiRight+ProLeft} vs Disagreed={AntiLeft+ProRight}
- **Level C** — Within Agreed: AntiRight vs ProLeft

**Data source**: Voxel-level NPZ files from `data/derivatives/postbypost/parcel_patterns/`  
(same files as `parcel_ispc_leftwing.ipynb` — no re-extraction needed).

In [ ]:
from pathlib import Path
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys

sys.path.insert(0, str(Path.cwd().parent / 'src'))

from yy_fmri_kit.event_isc.parcel_pattern_similarity import load_subject_patterns_postwise
from yy_fmri_kit.event_isc.parcel_isrsa import (
    make_behavioral_sim,
    merge_pattern_conditions,
    run_isrsa_both,
)
from yy_fmri_kit.event_isc.contrast import parcels_to_nifti
from yy_fmri_kit.visualization.pattern_analysis import (
    plot_similarity_matrices,
    plot_brain_behavior_scatter,
)
from statsmodels.stats.multitest import fdrcorrection
import yabplot as yab
import yabplot.data as ydata


## 1. Configuration

In [ ]:
ROOT             = Path('/root/to/project')  # CHANGE THIS to your local path
BEHAVIORAL_CSV   = ROOT / 'behavioral_analyses/data/250226/merged_behavioral_bids.csv'
POLARIZATION_CSV = ROOT / 'behavioral_analyses/data/250226/political_polarization_scored.csv'
NPZ_DIR          = ROOT / 'data/derivatives/postbypost/parcel_patterns'
ATLAS_NII        = ROOT / 'data/atlases/Schaefer2018_tf_2mm_400Parcels7Networks_plus_TianS3.dseg.nii.gz'
LABELS_TSV       = ROOT / 'data/atlases/Schaefer2018_400Parcels7Networks_plus_TianS3_labels.tsv'
OUTPUT_DIR       = ROOT / 'data/derivatives/rsa/parcel_isrsa_leftwing'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TYPES = ['AntiLeft', 'AntiRight', 'ProLeft', 'ProRight']
N_PERMS   = 1000
SEED      = 42
FDR_Q     = 0.05

## 2. Subject Selection

In [ ]:
behav_df = (
    pd.read_csv(BEHAVIORAL_CSV)
    .drop_duplicates(subset='bids_id', keep='first')
)
left_subs = sorted(behav_df.loc[behav_df['political_group'] == 'left', 'bids_id'].tolist())
print(f'Left-wing subjects (N={len(left_subs)}): {left_subs}')

## 3. Load NPZ Patterns (all 4 conditions)

In [ ]:
# Load post-wise patterns for all conditions.
# patterns_all[run_type] = (n_subjects, n_posts, n_brain_voxels)
# Atlas metadata is identical across conditions — read once.

patterns_all   = {}
subjects_by_rt = {}
vox_labels     = None
parcel_ids     = None
parcel_names   = None

for run_type in RUN_TYPES:
    pats, post_ids, vl, pids, pnames, loaded = load_subject_patterns_postwise(
        NPZ_DIR, left_subs, run_type
    )
    patterns_all[run_type]   = pats        # (n_subjects, n_posts, n_brain_voxels)
    subjects_by_rt[run_type] = loaded

    if vox_labels is None:
        vox_labels   = vl
        parcel_ids   = pids
        parcel_names = [str(n) for n in pnames]

    n_s, n_p, n_v = pats.shape
    print(f'  {run_type}: {n_s} subjects × {n_p} posts × {n_v} voxels')

# Use the subject list that is common across all conditions
common_subs = sorted(
    set.intersection(*[set(subjects_by_rt[rt]) for rt in RUN_TYPES])
)
print(f'\nCommon subjects across all conditions (N={len(common_subs)}): {common_subs}')

# Align all pattern arrays to common_subs order
for run_type in RUN_TYPES:
    loaded = subjects_by_rt[run_type]
    idx    = [loaded.index(s) for s in common_subs]
    patterns_all[run_type] = patterns_all[run_type][idx]

print(f'\nParcels: {len(parcel_ids)}  |  Brain voxels: {len(vox_labels)}')

## 4. Load affpol_thermo and Build Behavioral Similarity Matrix

In [ ]:
pol_df = (
    pd.read_csv(POLARIZATION_CSV)
    .assign(subject_code=lambda d: d['Code_tested'].str.replace(
        r'YY_PL_0*(\d+)', r'YY_PL_\1', regex=True))
    .drop_duplicates(subset='subject_code', keep='first')
    [['subject_code', 'affpol_thermo']]
)

pol_bids = pd.merge(
    pol_df,
    behav_df[['subject_code', 'bids_id']],
    on='subject_code', how='inner',
)
affiliation = dict(zip(pol_bids['bids_id'], pol_bids['affpol_thermo'].astype(float)))
print(f'affpol_thermo loaded for {len(affiliation)} subjects')

# Keep only subjects that have both NPZ data and behavioral scores
bb_subjects = [s for s in common_subs if s in affiliation]
bb_indices  = [common_subs.index(s) for s in bb_subjects]
print(f'Subjects with both neural and behavioral data (N={len(bb_subjects)}): {bb_subjects}')
print('\naffpol_thermo scores:')
for s in bb_subjects:
    print(f'  {s}: {affiliation[s]:.1f}')

# Slice pattern arrays to bb_subjects
patterns_bb = {rt: pats[bb_indices] for rt, pats in patterns_all.items()}

# N×N behavioral similarity matrix
beh_sim = make_behavioral_sim(affiliation, bb_subjects)
print(f'\nBehavioral similarity matrix: {beh_sim.shape}  '
      f'range [{beh_sim.min():.3f}, {beh_sim.max():.3f}]')

## 5. Brain-Map Helpers

In [ ]:
_lh_surf, _rh_surf = ydata.get_surface_paths('midthickness', 'bmesh')

ALL_VIEWS = [
    'left_lateral', 'left_medial', 'right_lateral', 'right_medial',
    'superior', 'inferior', 'anterior', 'posterior',
]

def brain_map(values, label, out_dir, *, cmap='coolwarm', vminmax=(None, None),
              nan_color=(0.92, 0.92, 0.92)):
    tmp = Path(tempfile.mktemp(suffix='.nii.gz'))
    parcels_to_nifti(values, parcel_names, ATLAS_NII, LABELS_TSV, tmp)
    lh_data, rh_data = yab.project_vol2surf(str(tmp), interpolation='nearest')
    tmp.unlink(missing_ok=True)
    lh_mesh, rh_mesh = yab.load_vertexwise_mesh(_lh_surf, _rh_surf, lh_data, rh_data)
    yab.plot_vertexwise(
        lh_mesh, rh_mesh, views=ALL_VIEWS,
        cmap=cmap, vminmax=list(vminmax), nan_color=nan_color,
        figsize=(1600, 800), display_type='static',
        export_path=str(out_dir / f'{label}.png'),
    )

S3_TO_S1 = {
    'HIP-rh': ['pHIP-rh'], 'AMY-rh': ['lAMY-rh','mAMY-rh'],
    'pTHA-rh': ['THA-DP-rh','THA-VP-rh'], 'aTHA-rh': ['THA-VA-rh','THA-DA-rh'],
    'NAc-rh': ['NAc-shell-rh','NAc-core-rh'], 'GP-rh': ['pGP-rh','aGP-rh'],
    'PUT-rh': ['aPUT-rh','pPUT-rh'], 'CAU-rh': ['aCAU-rh','pCAU-rh'],
    'HIP-lh': ['aHIP-lh','pHIP-lh'], 'AMY-lh': ['lAMY-lh','mAMY-lh'],
    'pTHA-lh': ['THA-DP-lh','THA-VP-lh'], 'aTHA-lh': ['THA-VA-lh','THA-DA-lh'],
    'NAc-lh': ['NAc-shell-lh','NAc-core-lh'], 'GP-lh': ['pGP-lh','aGP-lh'],
    'PUT-lh': ['aPUT-lh','pPUT-lh'], 'CAU-lh': ['aCAU-lh','pCAU-lh'],
}

def subcortical_map(values, label, out_dir, *, cmap='coolwarm', vminmax=(None, None),
                    nan_color=(0.92, 0.92, 0.92)):
    val_dict = {parcel_names[i]: float(values[i]) for i in range(len(values))}
    s1_dict  = {}
    for s1, s3s in S3_TO_S1.items():
        sub_vals = [val_dict[n] for n in s3s if n in val_dict]
        s1_dict[s1] = float(np.nanmean(sub_vals)) if sub_vals else float('nan')
    yab.plot_subcortical(
        data=s1_dict, atlas='tian2020_s1', views=ALL_VIEWS,
        cmap=cmap, vminmax=list(vminmax), nan_color=nan_color,
        figsize=(1600, 800), display_type='static',
        export_path=str(out_dir / f'{label}_subcortical.png'),
    )

def run_brain_maps(results_a, results_b, out_dir):
    """Full + FDR-sig cortical and subcortical maps for both approaches."""
    # shared colour scale across all conditions and both approaches
    all_r = np.concatenate([
        results_a[rt]['obs_r'] for rt in results_a
    ] + [
        results_b[rt]['obs_r'] for rt in results_b
    ])
    cmax    = float(np.nanmax(np.abs(all_r)))
    vminmax = (-cmax, cmax)

    for approach_label, results in [('A', results_a), ('B', results_b)]:
        for rt, r in results.items():
            obs_r    = r['obs_r']
            rejected = r['rejected']
            sig_r    = np.where(rejected, obs_r, np.nan)
            lbl = f"{approach_label}_{rt.lower()}"
            brain_map(obs_r, f'{lbl}_full', out_dir, cmap='coolwarm', vminmax=vminmax)
            subcortical_map(obs_r, f'{lbl}_full', out_dir, cmap='coolwarm', vminmax=vminmax)
            brain_map(sig_r, f'{lbl}_sig', out_dir, cmap='coolwarm', vminmax=vminmax)
            subcortical_map(sig_r, f'{lbl}_sig', out_dir, cmap='coolwarm', vminmax=vminmax)
            print(f"  {approach_label}/{rt}: {int(rejected.sum())} FDR-sig parcels")

print('Helpers ready.')

## 6. IS-RSA Runner and Save Helpers

In [ ]:
def run_level(level_patterns, out_dir):
    """
    Run IS-RSA (both approaches) for every condition in level_patterns.

    Parameters
    ----------
    level_patterns : {condition: (n_subjects, n_posts, n_brain_voxels)}
    out_dir        : output directory for this level

    Returns
    -------
    results_a, results_b : {condition: {obs_r, p_vals, null_dist, rejected, p_fdr}}
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    results_a, results_b = {}, {}

    for condition, pats in level_patterns.items():
        print(f'\nCondition: {condition}  '
              f'({pats.shape[0]} subjects × {pats.shape[1]} posts × {pats.shape[2]} voxels)')
        ra, rb = run_isrsa_both(
            pats, vox_labels, parcel_ids, beh_sim,
            n_perms=N_PERMS, seed=SEED, fdr_q=FDR_Q, verbose=True,
        )
        results_a[condition] = ra
        results_b[condition] = rb

    return results_a, results_b


def save_level(results_a, results_b, out_dir):
    """Save CSVs and null distributions for one level."""
    for approach_label, results in [('A', results_a), ('B', results_b)]:
        for condition, r in results.items():
            rows = []
            for i, pname in enumerate(parcel_names):
                rows.append({
                    'parcel'     : pname,
                    'rsa_r'      : round(float(r['obs_r'][i]), 6),
                    'p_perm'     : round(float(r['p_vals'][i]), 6) if not np.isnan(r['p_vals'][i]) else float('nan'),
                    'p_fdr'      : round(float(r['p_fdr'][i]),  6) if not np.isnan(r['p_fdr'][i])  else float('nan'),
                    'significant': bool(r['rejected'][i]),
                })
            pd.DataFrame(rows).to_csv(
                out_dir / f'approach{approach_label}_{condition}.csv', index=False
            )
            np.save(out_dir / f'approach{approach_label}_{condition}_null_dist.npy',
                    r['null_dist'])
    print(f'Saved to {out_dir}')


def summary_table(results_a, results_b):
    """Print a compact summary comparing both approaches across conditions."""
    rows = []
    for approach, results in [('A (mean)', results_a), ('B (post-wise)', results_b)]:
        for condition, r in results.items():
            rows.append({
                'approach'  : approach,
                'condition' : condition,
                'mean_r'    : round(float(np.nanmean(r['obs_r'])), 4),
                'max_r'     : round(float(np.nanmax(r['obs_r'])),  4),
                'n_sig_fdr' : int(r['rejected'].sum()),
            })
    return pd.DataFrame(rows).set_index(['approach', 'condition'])

print('Helpers ready.')

In [ ]:
def plot_sig_scatters_both(results_a, results_b, condition,
                           beh_sim, subjects, parcel_names, out_dir,
                           top_n_fallback=5):
    """
    For a single condition, generate similarity-matrix heatmaps and
    neural-vs-behavioral scatter plots for both approaches.

    Shows FDR-significant parcels.  If none are significant, falls back
    to the top `top_n_fallback` parcels by IS-RSA r.

    Saves:
      approach{A/B}_{condition}_{parcel}_simmat.png
      approach{A/B}_{condition}_{parcel}_scatter.png
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    for approach_label, r in [('A', results_a[condition]),
                               ('B', results_b[condition])]:
        neural_sim = r['neural_sim']   # (n_subjects, n_subjects, n_parcels)
        obs_r      = r['obs_r']
        rejected   = r['rejected']

        # Select parcels to plot
        sig_idx = np.where(rejected)[0]
        if len(sig_idx) == 0:
            print(f"  Approach {approach_label} / {condition}: "
                  f"no FDR-sig parcels — showing top {top_n_fallback} by r")
            sig_idx = np.argsort(obs_r)[::-1][:top_n_fallback]
            sig_idx = sig_idx[~np.isnan(obs_r[sig_idx])]
        else:
            sig_idx = sorted(sig_idx, key=lambda i: obs_r[i], reverse=True)

        print(f"  Approach {approach_label} / {condition}: "
              f"plotting {len(sig_idx)} parcel(s)")

        for p_idx in sig_idx:
            pname = parcel_names[p_idx]
            safe  = pname.replace('/', '_').replace(' ', '_')

            # Similarity matrices
            fig = plot_similarity_matrices(
                neural_sim, beh_sim, subjects,
                parcel_idx=p_idx, parcel_name=pname, run_type=condition,
                beh_label='affpol_thermo',
            )
            fig.savefig(
                out_dir / f'approach{approach_label}_{condition}_{safe}_simmat.png',
                dpi=150, bbox_inches='tight',
            )
            plt.show()

            # Scatter: neural vs behavioral similarity
            fig = plot_brain_behavior_scatter(
                neural_sim, beh_sim, subjects,
                parcel_idx=p_idx, parcel_name=pname, run_type=condition,
                beh_label='affpol_thermo',
            )
            fig.savefig(
                out_dir / f'approach{approach_label}_{condition}_{safe}_scatter.png',
                dpi=150, bbox_inches='tight',
            )
            plt.show()


def beh_sim_heatmap(beh_sim, subjects, out_dir, fname='behavioral_similarity.png',
                    title='Behavioral similarity matrix (affpol_thermo)'):
    """
    Heatmap of the full N×N behavioral similarity matrix.
    Saved once (shared across all levels and conditions).
    """
    fig, ax = plt.subplots(figsize=(8, 7))
    labels  = [s.replace('sub-', '') for s in subjects]
    im = ax.imshow(beh_sim, cmap='RdBu_r', vmin=beh_sim.min(), vmax=beh_sim.max(),
                   aspect='auto')
    plt.colorbar(im, ax=ax, shrink=0.8, label='Behavioral similarity')
    ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
    ax.set_yticklabels(labels, fontsize=7)
    ax.set_title(title, fontsize=11)
    plt.tight_layout()
    out_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_dir / fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved → {out_dir / fname}')

print('Visualisation helpers ready.')


In [ ]:
# Behavioral similarity matrix heatmap — saved once, shared across all analyses
beh_sim_heatmap(beh_sim, bb_subjects, OUTPUT_DIR)

---
## Level A — All 4 Conditions

IS-RSA run separately for AntiLeft, AntiRight, ProLeft, ProRight.

In [ ]:
OUT_A = OUTPUT_DIR / 'all_conditions'

print('=== Level A: all 4 conditions ===')
results_A_a, results_A_b = run_level(patterns_bb, OUT_A)
save_level(results_A_a, results_A_b, OUT_A)

In [ ]:
summary_table(results_A_a, results_A_b)

In [ ]:
# Top FDR-significant parcels per approach × condition
for approach_label, results in [('A (mean)', results_A_a), ('B (post-wise)', results_A_b)]:
    print(f'\n── Approach {approach_label} ──')
    for condition, r in results.items():
        sig_idx = np.where(r['rejected'])[0]
        if len(sig_idx) == 0:
            print(f'  {condition}: no FDR-significant parcels')
            continue
        top = sorted(sig_idx, key=lambda i: r['obs_r'][i], reverse=True)
        print(f'  {condition} ({len(top)} FDR-sig):')
        for i in top[:5]:
            print(f'    {parcel_names[i]:<55}  r={r["obs_r"][i]:.4f}')

In [ ]:
run_brain_maps(results_A_a, results_A_b, OUT_A)

In [ ]:
for condition in RUN_TYPES:
    print(f'\n─── {condition} ───')
    plot_sig_scatters_both(
        results_A_a, results_A_b, condition,
        beh_sim, bb_subjects, parcel_names, OUT_A,
    )

### Level A — Similarity Matrices & Scatter Plots

Neural vs. behavioral similarity matrices and scatter plots for each condition.  
FDR-significant parcels are shown; falls back to top 5 by IS-RSA r if none are significant.

---
## Level B — Agreed vs Disagreed *(H1 level)*

**Agreed** = AntiRight + ProLeft (content left-wing subjects agree with)  
**Disagreed** = AntiLeft + ProRight (content they disagree with)

In [ ]:
CONDITION_MAP_B = {
    'agreed'    : ['AntiRight', 'ProLeft'],
    'disagreed' : ['AntiLeft',  'ProRight'],
}
merged_B = merge_pattern_conditions(patterns_bb, CONDITION_MAP_B)
for cond, pats in merged_B.items():
    print(f'  {cond}: {pats.shape[0]} subjects × {pats.shape[1]} posts × {pats.shape[2]} voxels')

In [ ]:
OUT_B = OUTPUT_DIR / 'agreed_vs_disagreed'

print('=== Level B: agreed vs disagreed ===')
results_B_a, results_B_b = run_level(merged_B, OUT_B)
save_level(results_B_a, results_B_b, OUT_B)

In [ ]:
summary_table(results_B_a, results_B_b)

In [ ]:
for approach_label, results in [('A (mean)', results_B_a), ('B (post-wise)', results_B_b)]:
    print(f'\n── Approach {approach_label} ──')
    for condition, r in results.items():
        sig_idx = np.where(r['rejected'])[0]
        if len(sig_idx) == 0:
            print(f'  {condition}: no FDR-significant parcels')
            continue
        top = sorted(sig_idx, key=lambda i: r['obs_r'][i], reverse=True)
        print(f'  {condition} ({len(top)} FDR-sig):')
        for i in top[:5]:
            print(f'    {parcel_names[i]:<55}  r={r["obs_r"][i]:.4f}')

In [ ]:
run_brain_maps(results_B_a, results_B_b, OUT_B)

In [ ]:
for condition in ['agreed', 'disagreed']:
    print(f'\n─── {condition} ───')
    plot_sig_scatters_both(
        results_B_a, results_B_b, condition,
        beh_sim, bb_subjects, parcel_names, OUT_B,
    )

### Level B — Similarity Matrices & Scatter Plots

---
## Level C — Within Agreed: AntiRight vs ProLeft *(H2 level)*

- **AntiRight** — outgroup-derogating content (attacking the right)  
- **ProLeft** — ingroup-affirming content (supporting the left)

In [ ]:
within_agreed_C = {
    'AntiRight': patterns_bb['AntiRight'],
    'ProLeft'  : patterns_bb['ProLeft'],
}
OUT_C = OUTPUT_DIR / 'within_agreed'

print('=== Level C: within agreed ===')
results_C_a, results_C_b = run_level(within_agreed_C, OUT_C)
save_level(results_C_a, results_C_b, OUT_C)

In [ ]:
summary_table(results_C_a, results_C_b)

In [ ]:
for approach_label, results in [('A (mean)', results_C_a), ('B (post-wise)', results_C_b)]:
    print(f'\n── Approach {approach_label} ──')
    for condition, r in results.items():
        sig_idx = np.where(r['rejected'])[0]
        if len(sig_idx) == 0:
            print(f'  {condition}: no FDR-significant parcels')
            continue
        top = sorted(sig_idx, key=lambda i: r['obs_r'][i], reverse=True)
        print(f'  {condition} ({len(top)} FDR-sig):')
        for i in top[:5]:
            print(f'    {parcel_names[i]:<55}  r={r["obs_r"][i]:.4f}')

In [ ]:
run_brain_maps(results_C_a, results_C_b, OUT_C)

In [ ]:
for condition in ['AntiRight', 'ProLeft']:
    print(f'\n─── {condition} ───')
    plot_sig_scatters_both(
        results_C_a, results_C_b, condition,
        beh_sim, bb_subjects, parcel_names, OUT_C,
    )

### Level C — Similarity Matrices & Scatter Plots

---
## Approach A vs B: Comparison

Scatter plots of Approach A (mean-pattern) vs Approach B (post-wise) IS-RSA r per parcel,
for each level and each condition.

In [ ]:
def plot_ab_scatter(results_a, results_b, level_name, out_dir):
    conditions = list(results_a.keys())
    n_conds    = len(conditions)
    fig, axes  = plt.subplots(1, n_conds, figsize=(5 * n_conds, 5))
    if n_conds == 1:
        axes = [axes]

    for ax, cond in zip(axes, conditions):
        ra = results_a[cond]['obs_r']
        rb = results_b[cond]['obs_r']
        valid = ~(np.isnan(ra) | np.isnan(rb))
        ax.scatter(ra[valid], rb[valid], alpha=0.35, s=6, linewidths=0, color='#555')
        lim = max(np.abs(ra[valid]).max(), np.abs(rb[valid]).max()) * 1.1
        ax.plot([-lim, lim], [-lim, lim], 'k--', lw=0.8, alpha=0.5)
        ax.axhline(0, color='gray', lw=0.4); ax.axvline(0, color='gray', lw=0.4)
        ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
        ax.set_xlabel('Approach A (mean-pattern)', fontsize=9)
        ax.set_ylabel('Approach B (post-wise)',    fontsize=9)
        ax.set_title(cond, fontsize=11)
        r_corr = np.corrcoef(ra[valid], rb[valid])[0, 1]
        ax.text(0.05, 0.93, f'r = {r_corr:.3f}', transform=ax.transAxes, fontsize=9,
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))

    fig.suptitle(f'IS-RSA A vs B — {level_name}', fontsize=12, y=1.02)
    plt.tight_layout()
    plt.savefig(out_dir / 'AB_scatter.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_ab_scatter(results_A_a, results_A_b, 'Level A (all 4 conditions)', OUT_A)
plot_ab_scatter(results_B_a, results_B_b, 'Level B (agreed vs disagreed)', OUT_B)
plot_ab_scatter(results_C_a, results_C_b, 'Level C (within agreed)', OUT_C)

---
## Warmth IS-RSA — In-Group & Out-Group Warmth (Approach B)

**Behavioral models** (from `political_polarization_scored.csv`):
- `thermo_inparty` — in-group warmth rating (0–100 thermometer)
- `thermo_outparty` — out-group warmth rating (0–100 thermometer)

**Design**: For each of the 4 conditions separately (AntiLeft, AntiRight, ProLeft, ProRight):
1. Build N×N behavioral similarity matrix: `1 − |score_i − score_j| / 100`
2. Build N×N neural similarity matrix (Approach B: post-wise Pearson r, averaged over posts)
3. Correlate upper triangles → IS-RSA r per parcel
4. Permutation test: 1,000 subject-label shuffles on behavioral matrix
5. FDR correction (Benjamini–Hochberg, q=0.05)

**Output**: 8 brain maps (4 conditions × 2 models) + FDR-significant variants.  
**Output directory**: `OUTPUT_DIR / 'warmth_isrsa'`

In [ ]:
pol_df_warmth = (
    pd.read_csv(POLARIZATION_CSV)
    .assign(subject_code=lambda d: d['Code_tested'].str.replace(
        r'YY_PL_0*(\d+)', r'YY_PL_\1', regex=True))
    .drop_duplicates(subset='subject_code', keep='first')
    [['subject_code', 'thermo_inparty', 'thermo_outparty']]
)
warmth_bids = pd.merge(
    pol_df_warmth,
    behav_df[['subject_code', 'bids_id']],
    on='subject_code', how='inner',
)
inparty_scores  = dict(zip(warmth_bids['bids_id'], warmth_bids['thermo_inparty'].astype(float)))
outparty_scores = dict(zip(warmth_bids['bids_id'], warmth_bids['thermo_outparty'].astype(float)))
print(f'thermo_inparty  loaded: {len(inparty_scores)} subjects')
print(f'thermo_outparty loaded: {len(outparty_scores)} subjects')

In [ ]:
# Keep only subjects with neural data AND both warmth scores
warmth_subjects = [s for s in common_subs if s in inparty_scores and s in outparty_scores]
warmth_indices  = [common_subs.index(s) for s in warmth_subjects]
print(f'Subjects with neural + warmth data (N={len(warmth_subjects)}): {warmth_subjects}')

patterns_warmth = {rt: pats[warmth_indices] for rt, pats in patterns_all.items()}

# N×N behavioral similarity matrices (thermometer scale 0–100)
beh_sim_inparty  = make_behavioral_sim(inparty_scores,  warmth_subjects, scale_max=100.0)
beh_sim_outparty = make_behavioral_sim(outparty_scores, warmth_subjects, scale_max=100.0)

print(f'\nIn-party sim:   {beh_sim_inparty.shape}  '
      f'range [{beh_sim_inparty.min():.3f}, {beh_sim_inparty.max():.3f}]')
print(f'Out-party sim:  {beh_sim_outparty.shape}  '
      f'range [{beh_sim_outparty.min():.3f}, {beh_sim_outparty.max():.3f}]')

beh_sim_heatmap(beh_sim_inparty,  warmth_subjects, OUTPUT_DIR / 'warmth_isrsa',
                fname='behavioral_similarity_inparty.png',
                title='Behavioral similarity matrix (thermo_inparty)')
beh_sim_heatmap(beh_sim_outparty, warmth_subjects, OUTPUT_DIR / 'warmth_isrsa',
                fname='behavioral_similarity_outparty.png',
                title='Behavioral similarity matrix (thermo_outparty)')

In [ ]:
ISPC_SIG_CSV = ROOT / 'data/derivatives/parcel_ispc/leftwing/parcel_isc_B_significance.csv'

if ISPC_SIG_CSV.exists():
    _ispc_df = pd.read_csv(ISPC_SIG_CSV)
    ispc_sig_masks = {}
    for run_type in RUN_TYPES:
        _sig_names = set(
            _ispc_df.loc[(_ispc_df['condition'] == run_type) & _ispc_df['significant'].astype(bool),
                         'parcel_name'].astype(str)
        )
        _mask = np.array([name in _sig_names for name in parcel_names])
        ispc_sig_masks[run_type] = _mask
        print(f'{run_type}: {int(_mask.sum())} ISPC-significant parcels')
else:
    print(f'WARNING: ISPC significance CSV not found:\n  {ISPC_SIG_CSV}')
    print('IS-RSA will run on ALL parcels. Run parcel_ispc_leftwing.ipynb first.')
    ispc_sig_masks = None

In [ ]:
from yy_fmri_kit.event_isc.parcel_isrsa import compute_neural_similarity_b, permutation_test_isrsa

def run_warmth_isrsa(run_types, patterns_dict, beh_sims, out_dir, ispc_sig_masks=None):
    """
    Approach B IS-RSA for each condition × each behavioral model.
    Neural similarity computed once per condition and reused for both models.

    If ispc_sig_masks is provided ({run_type: bool array of shape (n_parcels,)}),
    parcels not significant in the ISPC analysis are set to NaN before IS-RSA,
    effectively excluding them from the correlation and permutation test.
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    results = {}
    for run_type in run_types:
        pats = patterns_dict[run_type]
        print(f'\n=== {run_type}: building Approach B neural similarity '
              f'({pats.shape[0]} subs × {pats.shape[1]} posts) ...')
        neural_sim = compute_neural_similarity_b(pats, vox_labels, parcel_ids)

        if ispc_sig_masks is not None and run_type in ispc_sig_masks:
            non_sig = ~ispc_sig_masks[run_type]
            neural_sim[:, :, non_sig] = np.nan
            print(f'  Restricted to {int(ispc_sig_masks[run_type].sum())} ISPC-significant parcels')

        results[run_type] = {}
        for model_name, beh_sim_m in beh_sims.items():
            print(f'  [{model_name}] {N_PERMS} permutations ...')
            obs_r, p_vals, null_dist = permutation_test_isrsa(
                neural_sim, beh_sim_m, n_perms=N_PERMS, seed=SEED, verbose=True,
            )
            valid_mask = ~np.isnan(p_vals)
            rejected   = np.zeros(len(p_vals), dtype=bool)
            p_fdr      = np.full(len(p_vals), np.nan)
            if valid_mask.sum() > 0:
                rej_v, p_fdr_v = fdrcorrection(p_vals[valid_mask], alpha=FDR_Q)
                rejected[valid_mask] = rej_v
                p_fdr[valid_mask]    = p_fdr_v
            print(f'  [{model_name}] {int(rejected.sum())} FDR-sig parcels  '
                  f'mean r={float(np.nanmean(obs_r)):.4f}  '
                  f'max r={float(np.nanmax(obs_r)):.4f}')
            results[run_type][model_name] = {
                'neural_sim': neural_sim,
                'obs_r'     : obs_r,
                'p_vals'    : p_vals,
                'null_dist' : null_dist,
                'rejected'  : rejected,
                'p_fdr'     : p_fdr,
            }
    return results

print('Warmth IS-RSA runner ready.')

In [ ]:
OUT_W = OUTPUT_DIR / 'warmth_isrsa'

BEH_SIMS_WARMTH = {
    'inparty' : beh_sim_inparty,
    'outparty': beh_sim_outparty,
}

results_warmth = run_warmth_isrsa(
    RUN_TYPES, patterns_warmth, BEH_SIMS_WARMTH, OUT_W,
    ispc_sig_masks=ispc_sig_masks,
)

In [ ]:
def save_warmth_results(results, out_dir):
    for run_type, models in results.items():
        for model_name, r in models.items():
            rows = []
            for i, pname in enumerate(parcel_names):
                rows.append({
                    'parcel'     : pname,
                    'rsa_r'      : round(float(r['obs_r'][i]), 6),
                    'p_perm'     : round(float(r['p_vals'][i]), 6) if not np.isnan(r['p_vals'][i]) else float('nan'),
                    'p_fdr'      : round(float(r['p_fdr'][i]),  6) if not np.isnan(r['p_fdr'][i])  else float('nan'),
                    'significant': bool(r['rejected'][i]),
                })
            pd.DataFrame(rows).to_csv(
                out_dir / f'B_{run_type}_{model_name}.csv', index=False
            )
            np.save(out_dir / f'B_{run_type}_{model_name}_null_dist.npy', r['null_dist'])
    print(f'Saved to {out_dir}')

save_warmth_results(results_warmth, OUT_W)

In [ ]:
def summary_warmth(results):
    rows = []
    for run_type, models in results.items():
        for model_name, r in models.items():
            rows.append({
                'condition' : run_type,
                'model'     : model_name,
                'mean_r'    : round(float(np.nanmean(r['obs_r'])), 4),
                'max_r'     : round(float(np.nanmax(r['obs_r'])),  4),
                'n_sig_fdr' : int(r['rejected'].sum()),
            })
    return pd.DataFrame(rows).set_index(['condition', 'model'])

summary_warmth(results_warmth)

In [ ]:
for run_type, models in results_warmth.items():
    print(f'\n── {run_type} ──')
    for model_name, r in models.items():
        sig_idx = np.where(r['rejected'])[0]
        if len(sig_idx) == 0:
            print(f'  {model_name}: no FDR-significant parcels')
            continue
        top = sorted(sig_idx, key=lambda i: r['obs_r'][i], reverse=True)
        print(f'  {model_name} ({len(top)} FDR-sig):')
        for i in top[:5]:
            print(f'    {parcel_names[i]:<55}  r={r["obs_r"][i]:.4f}')

In [ ]:
def run_warmth_brain_maps(results, out_dir):
    all_r = np.concatenate([
        r['obs_r'] for models in results.values() for r in models.values()
    ])
    cmax    = float(np.nanmax(np.abs(all_r)))
    vminmax = (-cmax, cmax)

    for run_type, models in results.items():
        for model_name, r in models.items():
            obs_r    = r['obs_r']
            rejected = r['rejected']
            sig_r    = np.where(rejected, obs_r, np.nan)
            lbl = f"B_{run_type.lower()}_{model_name}"
            brain_map(obs_r, f'{lbl}_full', out_dir, cmap='coolwarm', vminmax=vminmax)
            subcortical_map(obs_r, f'{lbl}_full', out_dir, cmap='coolwarm', vminmax=vminmax)
            brain_map(sig_r, f'{lbl}_sig', out_dir, cmap='coolwarm', vminmax=vminmax)
            subcortical_map(sig_r, f'{lbl}_sig', out_dir, cmap='coolwarm', vminmax=vminmax)
            print(f"  {run_type}/{model_name}: {int(rejected.sum())} FDR-sig parcels")

run_warmth_brain_maps(results_warmth, OUT_W)

---
### Warmth IS-RSA — Visualizations

1. **Interactive brain maps** — nilearn viewer with dropdowns (condition, model) and threshold slider
2. **Representational similarity matrices** — behavioral and neural matrices side by side, top-5 parcels per condition × model
3. **Scatter plots** — neural vs behavioral similarity per subject pair, top-5 parcels per condition × model

In [ ]:
from nilearn import plotting
import webbrowser

HTML_W = OUT_W / 'interactive_maps'
HTML_W.mkdir(exist_ok=True)

# Shared symmetric colour scale across all 8 maps
_all_obs_r = np.concatenate([
    r['obs_r'] for models in results_warmth.values() for r in models.values()
])
_r_abs_max = max(float(np.nanmax(np.abs(_all_obs_r))), 0.01)

# Pre-build NIfTIs and always save all 8 HTML files
_nifti_cache_w = {}
for _rt in RUN_TYPES:
    for _mn in ('inparty', 'outparty'):
        _obs = results_warmth[_rt][_mn]['obs_r']
        _tmp = Path(tempfile.mktemp(suffix='.nii.gz'))
        parcels_to_nifti(_obs, parcel_names, ATLAS_NII, LABELS_TSV, _tmp)
        _nifti_cache_w[(_rt, _mn)] = str(_tmp)
        plotting.view_img(
            str(_tmp), bg_img='MNI152', cmap='coolwarm',
            threshold=0.1, vmin=-_r_abs_max, vmax=_r_abs_max,
            symmetric_cmap=True,
            title=f'IS-RSA r — {_rt} / {_mn}',
        ).save_as_html(str(HTML_W / f'B_{_rt.lower()}_{_mn}.html'))
print(f'Saved 8 HTML maps → {HTML_W}  (colour scale: ±{_r_abs_max:.3f})')

# ── ipywidgets interactive viewer ─────────────────────────────────────────────
try:
    import ipywidgets as widgets

    _cond_w   = widgets.Dropdown(
        options=RUN_TYPES, value=RUN_TYPES[0], description='Condition:',
        layout=widgets.Layout(width='180px'),
    )
    _model_w  = widgets.Dropdown(
        options=['inparty', 'outparty'], value='inparty', description='Model:',
        layout=widgets.Layout(width='160px'),
    )
    _thresh_w = widgets.FloatSlider(
        min=0.0, max=round(_r_abs_max, 2), step=0.005, value=0.0,
        description='Threshold |r| ≥', readout_format='.3f',
        layout=widgets.Layout(width='420px'),
        style={'description_width': '120px'},
    )
    _open_btn = widgets.Button(
        description='Open in browser', button_style='info',
        layout=widgets.Layout(width='160px'),
        tooltip='Opens the current map as a full-screen HTML page in your browser',
    )
    _out_w = widgets.Output()

    def _update_warmth_map(*_):
        cond   = _cond_w.value
        model  = _model_w.value
        thresh = _thresh_w.value
        view   = plotting.view_img(
            _nifti_cache_w[(cond, model)],
            bg_img='MNI152', cmap='coolwarm',
            threshold=max(thresh, 1e-6),
            vmin=-_r_abs_max, vmax=_r_abs_max,
            symmetric_cmap=True,
            title=f'IS-RSA r — {cond} / {model}  (|r| ≥ {thresh:.3f})',
        )
        with _out_w:
            _out_w.clear_output(wait=True)
            display(view)

    def _open_browser(_):
        html_path = HTML_W / f'B_{_cond_w.value.lower()}_{_model_w.value}.html'
        webbrowser.open(str(html_path))

    _open_btn.on_click(_open_browser)

    # Display layout and render initial map before attaching observers —
    # prevents spurious triggers from the frontend sync firing _update_warmth_map
    # once per widget on first display (which would stack 3+ copies in _out_w).
    display(widgets.VBox([
        widgets.HBox([_cond_w, _model_w, _thresh_w, _open_btn]),
        _out_w,
    ]))
    _update_warmth_map()

    for _w in (_cond_w, _model_w, _thresh_w):
        _w.observe(_update_warmth_map, names='value')

except ImportError:
    from IPython.display import IFrame
    print('ipywidgets not available — displaying first map as IFrame.')
    display(IFrame(str(HTML_W / f'B_{RUN_TYPES[0].lower()}_inparty.html'), width='100%', height=500))

#### Representational Similarity Matrices — Top-5 Parcels by IS-RSA r

For each condition × model, the 5 parcels with the highest IS-RSA r.  
Left panel: behavioral similarity matrix (`thermo_inparty` or `thermo_outparty`).  
Right panel: neural similarity matrix (Approach B — post-wise Pearson r, averaged over posts).

In [ ]:
def plot_warmth_rdms(results, beh_sims, subjects, parcel_names, out_dir, top_n=5):
    out_dir.mkdir(parents=True, exist_ok=True)
    for run_type, models in results.items():
        for model_name, r in models.items():
            obs_r      = r['obs_r']
            neural_sim = r['neural_sim']
            beh_sim_m  = beh_sims[model_name]

            valid_idx = np.where(~np.isnan(obs_r))[0]
            top_idx   = valid_idx[np.argsort(obs_r[valid_idx])[::-1]][:top_n]

            print(f'\n─── {run_type} / {model_name} ───')
            for p_idx in top_idx:
                pname = parcel_names[p_idx]
                safe  = pname.replace('/', '_').replace(' ', '_')
                fig = plot_similarity_matrices(
                    neural_sim, beh_sim_m, subjects,
                    parcel_idx=p_idx, parcel_name=pname,
                    run_type=f'{run_type} · {model_name}',
                    beh_label=f'thermo_{model_name}',
                )
                fig.savefig(
                    out_dir / f'B_{run_type}_{model_name}_{safe}_rdm.png',
                    dpi=150, bbox_inches='tight',
                )
                plt.show()
                print(f'  {pname}  IS-RSA r={obs_r[p_idx]:.4f}')

RDM_DIR = OUT_W / 'rdm_plots'
plot_warmth_rdms(
    results_warmth, BEH_SIMS_WARMTH, warmth_subjects,
    parcel_names, RDM_DIR,
)

#### Scatter Plots — Neural vs Behavioral Similarity (Top-5 Parcels)

For each condition × model, the 5 parcels with the highest IS-RSA r.  
Each point is a subject pair (upper triangle of the N×N matrices).

In [ ]:
def plot_warmth_scatters(results, beh_sims, subjects, parcel_names, out_dir, top_n=5):
    out_dir.mkdir(parents=True, exist_ok=True)
    for run_type, models in results.items():
        for model_name, r in models.items():
            obs_r      = r['obs_r']
            neural_sim = r['neural_sim']
            beh_sim_m  = beh_sims[model_name]

            valid_idx = np.where(~np.isnan(obs_r))[0]
            top_idx   = valid_idx[np.argsort(obs_r[valid_idx])[::-1]][:top_n]

            print(f'\n─── {run_type} / {model_name} ───')
            for p_idx in top_idx:
                pname = parcel_names[p_idx]
                safe  = pname.replace('/', '_').replace(' ', '_')
                fig = plot_brain_behavior_scatter(
                    neural_sim, beh_sim_m, subjects,
                    parcel_idx=p_idx, parcel_name=pname,
                    run_type=f'{run_type} · {model_name}',
                    beh_label=f'thermo_{model_name}',
                )
                fig.savefig(
                    out_dir / f'B_{run_type}_{model_name}_{safe}_scatter.png',
                    dpi=150, bbox_inches='tight',
                )
                plt.show()
                print(f'  {pname}  IS-RSA r={obs_r[p_idx]:.4f}')

SCATTER_DIR = OUT_W / 'scatter_plots'
plot_warmth_scatters(
    results_warmth, BEH_SIMS_WARMTH, warmth_subjects,
    parcel_names, SCATTER_DIR,
)